# Conditional models and classifier-free guidance

Notebooks 01-03 use an **unconditional prior** and bring the observations in
at sampling time with SDA guidance. This notebook tries the other route
(`docs/06_conditional_models.md`): a **conditional model** that takes the
observations as extra input channels and learns `p(state | observations)`
directly.

On the structured synthetic fields of notebook 02 (two variables, A and B,
correlation 0.8 at every scale), we:

1. train a small conditional model that sees **noisy bands of A only**,
   with **observation dropout** (`null_probability`: one sample in five sees
   "nothing observed"), so the same network also learns the prior;
2. sample it as a plain conditional model (`guidance="conditional"`, M_3)
   and with **classifier-free guidance** (`guidance="cfg"`, M_4) at several
   weights `w`;
3. measure ensemble-mean error and ensemble spread on held-out windows,
   inside and outside the observed bands, for the observed A and the
   never-observed B.

No data bundle is needed. On one GPU the notebook takes about 20 minutes
(11 of them training); on a CPU the notebook trains a shorter, rougher model. This is a toy: the point
is the workflow and the diagnostics, not a verdict on CFG for ocean data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

import obottle_starter as obs
from obottle_starter.inference.metrics import spread_skill
from obottle_starter.training.data import StructuredDataset
from obottle_starter.training.obs_simulation import simulate_masks
from obottle_starter.training.train import train

ROOT = Path("..")
device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = OmegaConf.merge(
    OmegaConf.load(ROOT / "configs" / "train_base.yaml"),
    OmegaConf.load(ROOT / "configs" / "train_structured_conditional.yaml"),
    {
        "output_dir": "runs/notebook_cfg",
        "start": "auto",  # re-running the notebook continues the same run
    },
)
if device == "cpu":
    cfg.num_steps = 1000  # a rough model; the comparisons below get noisier
names = obs.channel_names(cfg)  # ['A', 'B']
T, H, W = cfg.time_length, cfg.H, cfg.W
print(OmegaConf.to_yaml(cfg.conditioning))

## What the model is trained on

For every training sample, `training/obs_simulation.py` draws a fresh band
mask for A, adds Gaussian noise (`noise_std: 0.1` model units), and builds
two extra input channels: the observed values (0 elsewhere) and the 0/1
mask. With `null_probability: 0.2`, a fifth of the samples get all-zero
channels instead: the **null condition**. That is what lets CFG ask the
same network for an unconditional estimate.

The model learns the observing system it is trained on, so the held-out
test cases below use the same kind of masks (the first exercise at the end
changes that). They come from `split=2` of the generated data, which
neither training (`split=0`) nor validation (`split=1`) draws from.

In [ ]:
held_out = StructuredDataset(cfg, split=2)
train_obs = cfg.conditioning.training_observations


def test_case(i):
    # Held-out truth {A, B: (T, H, W)}, frame times, and a band mask for A.
    sample = held_out[i]
    fields = sample["target"].numpy().reshape(len(names), T, H, W)
    hours = sample["time_offsets"].numpy().round().astype("int64")
    times = np.datetime64("2012-01-01T00") + hours.astype("timedelta64[h]")
    torch.manual_seed(i)
    mask = simulate_masks(train_obs, ["A"], 1, T, H, W, "cpu")
    return dict(zip(names, fields)), times, mask[0, 0].reshape(T, H, W).numpy()


truth, times, mask = test_case(0)
print("frame offsets (h):", (times - times[0]).astype(int))
fig, axes = plt.subplots(3, T, figsize=(3 * T, 9), squeeze=False)
for t in range(T):
    kw = dict(origin="lower", cmap="RdBu_r", vmin=-3, vmax=3)
    axes[0, t].imshow(truth["A"][t], **kw)
    axes[0, t].set_title(f"A truth, frame {t}")
    axes[1, t].imshow(np.where(mask[t], truth["A"][t], np.nan), **kw)
    axes[1, t].set_title(f"A observed ({100 * mask[t].mean():.0f}%)")
    axes[2, t].imshow(truth["B"][t], **kw)
    axes[2, t].set_title("B truth (never observed)")
plt.tight_layout()

## Train

The log prints the training loss and, every `validation.every` steps, the
held-out denoising loss by noise level. For a conditional model the
validation windows get simulated observations too. If you re-run the
notebook, `start: auto` resumes the run in `runs/notebook_cfg` instead of
starting again.

In [ ]:
out_dir = train(cfg)
net, model_cfg = obs.load_checkpoint(
    out_dir / "checkpoints" / "checkpoint_latest.pt", device
)

## Sampling with different guidance weights

Classifier-free guidance runs the network twice per step, with the
observations and with the null condition, and extrapolates:

    x̂_w = x̂_null + w (x̂_cond − x̂_null)

* `w = 0`: the null branch only, i.e. the **prior** (given the frame times);
* `w = 1`: the plain **conditional** model (M_3). `assimilate` runs it with
  `guidance="conditional"`, one network pass per step;
* `w > 1`: CFG (M_4) pushes harder towards the observations.

`sigma_y` is required by `assimilate`, but the conditional routes never use
it as an assumed error: the observations reach the model only through its
inputs. Here it only sets the noise of the synthetic test observations, so
we keep it equal to the training `noise_std`.

The output's `observations` variable holds the noisy values the model was
given (NaN elsewhere), as in notebook 01.

In [ ]:
def sample(w, **kw):
    if w == 1.0:
        return obs.assimilate(net, model_cfg, guidance="conditional", **kw)
    return obs.assimilate(net, model_cfg, guidance="cfg", cfg_weight=w, **kw)


def run(case, w, n_members=16, seed=0):
    truth, times, mask = case
    return sample(
        w,
        times=times,
        center_lon=0.0,
        truth=truth,
        masks={"A": mask},
        sigma_y={"A": float(train_obs.noise_std)},
        n_members=n_members,
        steps=32,
        seed=seed,
        verbose=False,
    )


SHOW = [0.0, 1.0, 2.0, 4.0]
case = test_case(0)
posts = {w: run(case, w) for w in SHOW}

In [ ]:
t = 1
kw = dict(origin="lower", cmap="RdBu_r", vmin=-3, vmax=3)
first = posts[1.0]
fig, axes = plt.subplots(len(SHOW) + 1, 6, figsize=(18, 3 * (len(SHOW) + 1)))
top = [
    (first.truth.sel(channel="A")[t], "A truth"),
    (first.observations.sel(channel="A")[t], "A observations (noisy)"),
    (first.truth.sel(channel="B")[t], "B truth"),
]
for ax, (field, title) in zip(axes[0], top):
    ax.imshow(field, **kw)
    ax.set_title(f"{title}, frame {t}")
for ax in axes[0, 3:]:
    ax.axis("off")
for row, w in zip(axes[1:], SHOW):
    s = posts[w].samples
    panels = []
    for c in names:
        panels += [
            (s.sel(channel=c)[0, t], f"{c} member 0", kw),
            (s.sel(channel=c).mean("sample")[t], f"{c} mean", kw),
            (s.sel(channel=c).std("sample")[t], f"{c} spread",
             dict(origin="lower", cmap="viridis", vmin=0, vmax=1.2)),
        ]
    for ax, (field, title, k) in zip(row, panels):
        ax.imshow(field, **k)
        ax.set_title(f"w = {w:g}: {title}")
plt.tight_layout()

## Error and spread over many held-out windows

One window is one case. Below, each weight is run on `N_WINDOWS` held-out
windows, each with its own band mask. For each variable we pool:

* **ensemble-mean RMSE** inside the A bands and outside them (for B, "inside"
  means under the bands where A was observed);
* **ensemble spread** (the members' standard deviation) in the same places;
* the **error / spread ratio** from `metrics.spread_skill`: about 1 for a
  calibrated ensemble, above 1 when the ensemble is over-confident.

The fields have unit variance, so an RMSE of 1 is no skill, and the A
observations themselves are off by 0.1 on average.

In [ ]:
N_WINDOWS = 16 if device == "cuda" else 4
WEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
# Pooled squared errors and variances, keyed by (weight, variable, region).
sq = {}
for i in range(N_WINDOWS):
    case = test_case(i)
    truth, _, mask = case
    for w in WEIGHTS:
        post = run(case, w, n_members=8, seed=i)
        for c in names:
            s = post.samples.sel(channel=c).values
            for region, sel in [("in bands", mask), ("outside", ~mask)]:
                ss = spread_skill(s, truth[c], sel)
                acc = sq.setdefault((w, c, region), np.zeros(2))
                acc += [ss["rmse"] ** 2, ss["spread"] ** 2]
    print(f"window {i + 1}/{N_WINDOWS} done")

M = 8
stats = {k: np.sqrt(v / N_WINDOWS) for k, v in sq.items()}  # (rmse, spread)
header = ("w", "var", "region", "RMSE", "spread", "err/spread")
print("\n{:>4} {:>3} {:>9} {:>6} {:>6} {:>10}".format(*header))
for w in WEIGHTS:
    for c in names:
        for region in ("in bands", "outside"):
            err, spread = stats[(w, c, region)]
            ratio = err / (spread * np.sqrt((M + 1) / M))
            print(f"{w:4g} {c:>3} {region:>9} {err:6.3f} {spread:6.3f} {ratio:10.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, c in zip(axes, names):
    for region, color in [("in bands", "C0"), ("outside", "C1")]:
        err = [stats[(w, c, region)][0] for w in WEIGHTS]
        spread = [stats[(w, c, region)][1] for w in WEIGHTS]
        ax.plot(WEIGHTS, err, "o-", color=color, label=f"RMSE, {region}")
        ax.plot(WEIGHTS, spread, "s--", color=color, label=f"spread, {region}")
    if c == "A":
        ax.axhline(float(train_obs.noise_std), color="k", lw=0.5, ls=":",
                   label="observation noise")
    ax.axvline(1.0, color="grey", lw=0.5)
    ax.set_xlabel("CFG weight w  (0 = prior, 1 = conditional model)")
    ax.set_title(f"{c} ({'observed in bands' if c == 'A' else 'never observed'})")
    ax.legend(fontsize=8)
axes[0].set_ylabel("model units (field std = 1)")
plt.tight_layout()

## Reading the result

When we ran it (5000 steps on one L40S, 16 windows, 8 members; your numbers
will differ somewhat):

| w | A in bands: RMSE / spread | A outside | B under the bands | B elsewhere |
|---|---|---|---|---|
| 0 (prior) | 1.03 / 0.94 | 1.01 / 0.95 | 1.04 / 0.91 | 1.03 / 0.92 |
| 1 (conditional) | 0.14 / 0.19 | 0.76 / 0.69 | 0.64 / 0.53 | 0.89 / 0.76 |
| 2 | 0.16 / 0.10 | 0.78 / 0.64 | 0.67 / 0.46 | 0.90 / 0.70 |
| 4 | 0.19 / 0.08 | 0.86 / 0.63 | 0.73 / 0.46 | 0.97 / 0.68 |

* **`w = 0` is the prior.** No skill (RMSE about 1, the field std), but the
  spread matches the error: an honest "I don't know".
* **`w = 1` uses the observations well.** A in the bands is recovered to
  about the observation noise (0.1). Outside the bands the error drops too,
  because the fields are smooth and drift between frames, so a band in one
  frame says something about its neighbours in the others. **B was never
  observed**, yet under the A bands its error is 0.64. That is close to the
  best possible: with correlation 0.8, even a perfect A leaves
  `sqrt(1 - 0.8²) = 0.6` of B unexplained. The model learned the
  cross-variable relationship from the training data alone.
* **`w > 1` sharpens, but does not improve.** The spread keeps shrinking,
  while the error grows slowly. At `w = 4` the spread of A in the bands is
  below the observation noise itself, and error/spread is about 2: the
  ensemble is over-confident. In this toy problem CFG buys sharper-looking
  members at the cost of calibration, not accuracy. Whether that holds for
  ocean data, other observing systems or better-trained models is an open
  question, not a conclusion.
* **Caveats.** One training run, whose validation loss was still falling
  at the end (try `num_steps: 20000`). Test masks are the kind the model was
  trained on. In the bands at `w = 1` the ensemble is somewhat
  over-dispersed (error/spread 0.7).

## Exercises: change this file for this experiment

The first group needs only config or notebook changes. The last needs new
code; it marks the boundary between what the conditioning supports
(observation *maps*) and other context.

| Experiment | What to change |
|---|---|
| Test on an observing system the model never saw | In `test_case`, replace the band mask by `obottle_starter.inference.osse.random_masks(["A"], (T, H, W), 0.2, seed=i)["A"]` (scattered cells), or use wider bands. Compare with the in-distribution numbers above. |
| Observe B too | `conditioning.observe: [A, B]` in `configs/train_structured_conditional.yaml` (and a `stripe_width` per variable if you like); pass `masks={"A": ..., "B": ...}`. Use a new `output_dir`: the input channels change. |
| Does the mask channel matter? | `mask_channel: false`, retrain, compare. |
| How much observation dropout? | `null_probability: 0.05` or `0.5`. With `0`, `guidance="cfg"` refuses the model: the null branch was never trained. |
| Conditional model against prior + SDA | Train `configs/train_structured.yaml` (the same data, unconditional) and run `obs.assimilate(..., guidance="mmps", sigma_y={"A": 0.1})` on the same `test_case(i)` windows. |
| Fine-tune instead of training from scratch | Train the unconditional model above first, then add `init_from=<its checkpoint_latest.pt>` here; the new input channels start at zero (`core/model.py`, `load_weights_padded`). |
| **Needs code:** tell the model the noise level | Values and masks do not say how noisy the observations are. Add a channel holding `noise_std` at observed cells: draw `noise_std` per sample in `training_condition` (`training/obs_simulation.py`), add the channel in `observation_channels` and `build_obs_condition` (`core/obs_conditioning.py`), and build it from `sigma_y` in `assimilate` (`inference/assimilate.py`). |
| **Needs code:** other context (wind, a coarse product, a forcing field) | The dataset must return it (`training/data.py`), it must become extra input channels the same way, and inference must supply it. Observation *maps* of model variables are supported; anything else is a new input. |

See `docs/06_conditional_models.md` for the background and the open
questions, and `tests/test_conditional.py` for what is checked
automatically (that CFG with `w = 1` equals the conditional model, for
example).